In [0]:
%sql
SELECT * FROM read_files('/Volumes/happiness/landing/archivos/WGI/wgi_va.csv', format => 'csv', header => true) LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE happiness.bronze.wgi
COMMENT 'Bronze WGI (World Governance Indicators): una fila por pais + dimension de gobernanza + anio. 100% crudo, todas las columnas tal cual el archivo (incluidas las ~30 columnas de fuentes individuales), sin castear ni filtrar (ver D10). Union de 4 CSV (va, pv, ge, rl) exportados manualmente del Excel oficial. Valores faltantes como ".." (convencion propia de este dataset, distinta del "null" de WDI) - se dejan tal cual.'
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
)
AS
SELECT
  *,
  current_timestamp() AS _ingest_ts,
  'wgidataset_with_sourcedata-2026.xlsx' AS _source
FROM read_files(
  '/Volumes/happiness/landing/archivos/WGI/wgi_*.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false
);

In [0]:
%sql
DESCRIBE happiness.bronze.wgi;

In [0]:
%sql
SELECT `Governance dimension` AS dimension,
       COUNT(*) AS filas,
       COUNT(DISTINCT `Economy (code)`) AS entidades,
       MIN(Year) AS anio_min,
       MAX(Year) AS anio_max
FROM happiness.bronze.wgi
GROUP BY `Governance dimension`
ORDER BY dimension;